# 14_Statistics_in_ML

Machine learning models feel like a separate world from hypothesis testing, but almost every step underneath is statistics wearing a different hat — means, variances, distributions, covariance, probability. This notebook walks through where statistics actually shows up across a typical ML pipeline: cleaning, missing values, outliers, feature engineering, scaling, model evaluation, a few core algorithms, clustering, PCA, and recommendation systems.

## 01. Data Cleaning

**What it is**: The process of catching bad, inconsistent, or invalid entries in a dataset before it goes anywhere near a model.

**How statistics is used**: Descriptive statistics — min, max, mean, standard deviation, value counts — are the first line of defense. If a "age" column has a mean of 40 but a max of 999, that's a statistical red flag before it's a business one. Frequency tables catch inconsistent category labels (e.g. "NY", "New York", "ny" all meaning the same thing).

**Business Example**: A retailer's transaction log has a few rows with negative purchase amounts and a handful of ages listed as 150 — clearly data entry errors that need flagging before analysis.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'age': [25, 34, 150, 29, -5, 41, 38],
    'purchase_amount': [50, 120, 75, -20, 300, 90, 60]
})

print(df.describe())

invalid_age = df[(df['age'] < 0) | (df['age'] > 100)]
invalid_amount = df[df['purchase_amount'] < 0]
print(invalid_age)
print(invalid_amount)

              age  purchase_amount
count    7.000000         7.000000
mean    44.571429        96.428571
std     48.931439        99.612344
min     -5.000000       -20.000000
25%     27.000000        55.000000
50%     34.000000        75.000000
75%     39.500000       105.000000
max    150.000000       300.000000
   age  purchase_amount
2  150               75
4   -5              300
   age  purchase_amount
3   29              -20


**Interpretation**: The summary statistics immediately show a max age of 150 and a negative minimum purchase amount — both statistically implausible and worth flagging or correcting before the data reaches a model.

## 02. Missing Value Handling

**What it is**: Deciding what to do with gaps in the data — drop them, fill them, or model around them.

**How statistics is used**: The choice of imputation is a statistical decision. Mean imputation preserves the average but shrinks variance; median is safer with skewed data or outliers; mode works for categorical fields. Understanding whether data is missing completely at random, at random, or not at random (MCAR/MAR/MNAR) — usually checked by looking at correlations between missingness and other variables — determines whether imputation is even appropriate.

**Business Example**: A customer survey has some missing income values. Filling them with the mean would work if income is missing randomly, but if higher earners tend to skip the question, a simple mean fill would bias the results downward.

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'income': [45000, 52000, np.nan, 61000, np.nan, 48000, 75000]
})

mean_fill = df['income'].fillna(df['income'].mean())
median_fill = df['income'].fillna(df['income'].median())

print(df['income'].mean(), df['income'].median())
print(mean_fill.tolist())
print(median_fill.tolist())

56200.0 52000.0
[45000.0, 52000.0, 56200.0, 61000.0, 56200.0, 48000.0, 75000.0]
[45000.0, 52000.0, 52000.0, 61000.0, 52000.0, 48000.0, 75000.0]


**Interpretation**: Mean and median fill give slightly different results here because the distribution isn't perfectly symmetric — with skewed income data, median imputation is usually the safer default since it's less pulled around by extreme values.

## 03. Outlier Detection

**What it is**: Identifying data points that fall far outside the normal range of a variable.

**How statistics is used**: Two common statistical approaches — the Z-score method (flag anything beyond ~3 standard deviations from the mean) and the IQR method (flag anything beyond 1.5x the interquartile range from Q1/Q3). Both rely directly on distributional statistics rather than arbitrary cutoffs.

**Business Example**: A bank monitoring transaction amounts wants to flag unusually large transactions that might indicate fraud, without manually setting a fixed dollar threshold.

In [3]:
import numpy as np
import pandas as pd

transactions = pd.Series([120, 85, 95, 110, 130, 90, 105, 5000, 100, 115])

q1 = transactions.quantile(0.25)
q3 = transactions.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = transactions[(transactions < lower_bound) | (transactions > upper_bound)]
print(lower_bound, upper_bound)
print(outliers)

62.5 152.5
7    5000
dtype: int64


**Interpretation**: The 5000 transaction falls well outside the IQR bounds, flagging it as a statistical outlier worth a closer look — exactly the kind of value that could distort a model's learned patterns if left untreated.

## 04. Feature Engineering

**What it is**: Creating new input variables from existing data to help a model pick up on patterns it wouldn't see otherwise.

**How statistics is used**: Aggregation statistics (mean, count, sum per group) turn raw transactional data into meaningful features. Quantile-based binning turns continuous variables into categories. Correlation analysis helps decide which engineered features actually add signal versus redundant noise.

**Business Example**: An online retailer wants to turn raw order history into a feature like "average order value per customer" and "order frequency," which are far more predictive of churn than the raw transaction log.

In [4]:
import pandas as pd

orders = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2, 3, 3, 3, 3],
    'order_value': [50, 80, 60, 200, 150, 30, 40, 35, 45]
})

customer_features = orders.groupby('customer_id')['order_value'].agg(['mean', 'count', 'std'])
print(customer_features)

                   mean  count        std
customer_id                              
1             63.333333      3  15.275252
2            175.000000      2  35.355339
3             37.500000      4   6.454972


**Interpretation**: Each customer now has a mean order value, order count, and variability — statistics computed on the raw transaction log, but far more useful to a model than the individual order rows themselves.

## 05. Feature Scaling

**What it is**: Transforming features so they're on comparable scales, which matters for distance-based and gradient-based algorithms.

**How statistics is used**: Standardization subtracts the mean and divides by the standard deviation, turning every feature into a distribution with mean 0 and standard deviation 1 — a direct application of the z-score.

**Business Example**: A credit scoring model uses both "annual income" (ranging in the tens of thousands) and "number of credit inquiries" (ranging 0-10). Without scaling, income would dominate any distance-based calculation just because of its larger numeric range.

In [6]:
!pip install scikit-learn statsmodels pandas numpy scipy matplotlib

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -------- ------------------------------- 1.8/8.3 MB 15.3 MB/s eta 0:00:01
   ------------ --------------------------- 2.6/8.3 MB 7.4 MB/s eta 0:00:01
   ---------------- ----------------------- 3.4/8.3 MB 5.5 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.3 MB 5.1 MB/s eta 0:00:01
   --------------------- ------------------ 4.5/8.3 MB 4.5 MB/s eta 0:00:01
   --------------------- ------------------ 4.5/8.3 MB 4.5 MB/s eta 0:00:01
   ---------------------- ----------------- 4.7/8.3 MB 3.4 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.3 MB 3.1 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.3 MB 2.8 MB/s eta 0:00:02
   -------------------------- ------------- 5.5/8.3 MB 2.7 MB/s eta 0:00:02
   --------------------------- ------------ 5.8/8.3 MB 2.6 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 2.5 MB/s eta 0:00:01
   ---------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

data = np.array([
    [55000, 2],
    [72000, 5],
    [48000, 1],
    [90000, 3]
])

scaler = StandardScaler()
scaled = scaler.fit_transform(data)

print(scaler.mean_, scaler.scale_)
print(scaled)

[6.625e+04 2.750e+00] [1.62538457e+04 1.47901995e+00]
[[-0.69214389 -0.50709255]
 [ 0.35376243  1.52127766]
 [-1.1228112  -1.18321596]
 [ 1.46119266  0.16903085]]


**Interpretation**: After scaling, both columns sit on the same footing — a difference of one standard deviation means the same thing for income as it does for credit inquiries, which is exactly what distance-based models need.

## 06. Data Normalization

**What it is**: Rescaling features into a fixed range, typically 0 to 1, rather than standardizing around the mean.

**How statistics is used**: Min-max normalization uses the minimum and maximum of a variable directly, which is a simpler statistical transformation than standardization but sensitive to outliers since a single extreme value stretches the whole scale.

**Business Example**: A recommendation model blends a "popularity score" (ranging from 0 to a few million views) with a "recency score" (ranging 0-30 days) and needs both on the same 0-1 scale before combining them.

In [8]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

data = np.array([
    [1000000, 2],
    [50000, 15],
    [2000000, 1],
    [10000, 28]
])

scaler = MinMaxScaler()
normalized = scaler.fit_transform(data)
print(normalized)

[[0.49748744 0.03703704]
 [0.0201005  0.51851852]
 [1.         0.        ]
 [0.         1.        ]]


**Interpretation**: Every value now sits between 0 and 1, letting popularity and recency contribute proportionally to a combined score instead of popularity swamping everything by sheer magnitude.

## 07. Model Evaluation

**What it is**: Measuring how well a trained model actually performs, not just on training data but in a way that generalizes.

**How statistics is used**: Precision, recall, F1, and accuracy are all statistics computed from a confusion matrix. Cross-validation relies on resampling statistics to estimate how much performance varies across different splits of the data. Comparing two models statistically (not just eyeballing accuracy) often calls for something like a paired t-test on cross-validation scores.

**Business Example**: A churn prediction model needs to be evaluated not just on overall accuracy, but on whether it's actually catching the customers who are about to leave (recall) without flagging too many loyal customers as risks (precision).

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = [0, 1, 1, 0, 1, 0, 1, 1, 0, 0]
y_pred = [0, 1, 0, 0, 1, 0, 1, 1, 1, 0]

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

[[4 1]
 [1 4]]
              precision    recall  f1-score   support

           0       0.80      0.80      0.80         5
           1       0.80      0.80      0.80         5

    accuracy                           0.80        10
   macro avg       0.80      0.80      0.80        10
weighted avg       0.80      0.80      0.80        10



**Interpretation**: The classification report breaks performance down by class rather than a single accuracy number, which matters a lot when the business cost of missing a churner is very different from the cost of a false alarm.

## 08. Linear Regression

**What it is**: A model that fits a straight-line relationship between input variables and a continuous outcome.

**How statistics is used**: Linear regression is statistics end to end — the coefficients are estimated by minimizing squared error (least squares), R² measures how much variance the model explains, and p-values on each coefficient test whether that variable's relationship with the outcome is likely real or just noise. The whole method leans on assumptions like linearity, homoscedasticity, and normally distributed residuals.

**Business Example**: A real estate company wants to understand how much each additional square foot adds to a home's price, and whether that relationship is statistically reliable or could be due to chance in their sample.

In [10]:
import numpy as np
import statsmodels.api as sm

sqft = np.array([1200, 1500, 1800, 2000, 2400, 2600, 3000])
price = np.array([210000, 250000, 285000, 310000, 360000, 400000, 445000])

X = sm.add_constant(sqft)
model = sm.OLS(price, X).fit()

print(model.params)
print(model.pvalues)
print(model.rsquared)

[50384.61538462   131.53846154]
[6.99356232e-04 1.49254289e-07]
0.9971355498721227


**Interpretation**: The coefficient on square footage tells us the price increase per square foot, the p-value being far below 0.05 says that relationship isn't due to chance, and an R² close to 1 means square footage alone explains most of the price variation in this sample.

## 09. Logistic Regression

**What it is**: A model for predicting the probability of a binary outcome (yes/no, churn/stay, fraud/legitimate).

**How statistics is used**: Instead of least squares, logistic regression uses maximum likelihood estimation to find coefficients. Those coefficients translate into odds ratios — a direct statistical statement of how much a one-unit change in a feature multiplies the odds of the outcome. Wald tests give p-values for each coefficient, just like in linear regression.

**Business Example**: A bank wants to know how much each additional missed payment increases the odds that a loan defaults, and whether that effect is statistically significant.

In [11]:
import numpy as np
import statsmodels.api as sm

missed_payments = np.array([0, 1, 2, 0, 3, 1, 4, 0, 2, 3, 1, 2, 0, 4, 1])
defaulted = np.array([0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0])

X = sm.add_constant(missed_payments)
model = sm.Logit(defaulted, X).fit(disp=0)

print(model.params)
print(model.pvalues)
print(np.exp(model.params))

[-3.89577707  2.46448941]
[0.0556781  0.04940972]
[ 0.02032757 11.7574774 ]


**Interpretation**: The exponentiated coefficient is the odds ratio for one additional missed payment — a value well above 1 means each missed payment substantially increases the odds of default, and the p-value tells us whether that effect is statistically dependable.

## 10. Naive Bayes

**What it is**: A classification algorithm built directly on Bayes' theorem, assuming features are independent given the class.

**How statistics is used**: The "naive" part comes from treating each feature's probability distribution independently, then multiplying them together. Gaussian Naive Bayes literally assumes each feature is normally distributed within each class and estimates the mean and variance of that distribution to compute probabilities.

**Business Example**: An email provider wants to classify messages as spam or not spam based on features like word frequency, using probability estimates rather than a hard rule-based filter.

In [12]:
import numpy as np
from sklearn.naive_bayes import GaussianNB

X = np.array([
    [5, 1], [4, 1], [6, 0], [1, 5], [2, 6], [1, 4]
])
y = np.array(['spam', 'spam', 'spam', 'not_spam', 'not_spam', 'not_spam'])

model = GaussianNB()
model.fit(X, y)

new_email = np.array([[4, 2]])
print(model.predict(new_email))
print(model.predict_proba(new_email))

['spam']
[[1.52299844e-08 9.99999985e-01]]


**Interpretation**: The predicted probabilities come straight from the estimated normal distributions per class per feature — the model isn't just guessing a label, it's comparing how likely this combination of feature values is under each class's statistical profile.

## 11. Clustering

**What it is**: Grouping data points together based on similarity, without any labeled outcome to predict.

**How statistics is used**: K-means minimizes within-cluster variance — literally the sum of squared distances from each point to its cluster's mean. Distance metrics themselves (Euclidean distance) come from basic statistics. Evaluating cluster quality, like the silhouette score, is a statistical comparison of within-cluster versus between-cluster distances.

**Business Example**: A retailer wants to segment customers into groups based on spending habits and visit frequency, without predefining what those segments should look like.

In [13]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = np.array([
    [10, 2], [12, 3], [11, 1],
    [50, 40], [55, 42], [52, 38],
    [90, 85], [88, 80], [92, 82]
])

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)

print(labels)
print(kmeans.cluster_centers_)
print(silhouette_score(X, labels))

[2 2 2 0 0 0 1 1 1]
[[52.33333333 40.        ]
 [90.         82.33333333]
 [11.          2.        ]]
0.934688024612955


**Interpretation**: The cluster centers are just the mean position of each group, and a silhouette score close to 1 tells us the three customer segments are statistically well-separated rather than blending into each other.

## 12. Principal Component Analysis (PCA)

**What it is**: A technique for reducing the number of features in a dataset while keeping as much of the original information as possible.

**How statistics is used**: PCA is built entirely on the covariance matrix of the data. It finds the eigenvectors of that matrix — the directions of maximum variance — and reprojects the data onto those directions. The explained variance ratio tells you exactly how much of the original statistical spread each new component captures.

**Business Example**: A company has survey data with 20 correlated questions about customer satisfaction and wants to reduce that down to 2-3 underlying dimensions without losing much information.

In [14]:
import numpy as np
from sklearn.decomposition import PCA

np.random.seed(42)
base = np.random.normal(0, 1, 100)
X = np.column_stack([
    base + np.random.normal(0, 0.1, 100),
    base * 2 + np.random.normal(0, 0.1, 100),
    np.random.normal(0, 1, 100)
])

pca = PCA(n_components=2)
transformed = pca.fit_transform(X)

print(pca.explained_variance_ratio_)
print(pca.components_)

[0.84656518 0.15153734]
[[ 0.43618179  0.8954588  -0.08887622]
 [ 0.03347482  0.08255185  0.99602441]]


**Interpretation**: The explained variance ratio shows the first component alone captures most of the spread in the data — because two of the three original features were strongly correlated, PCA compressed that redundancy into a single dimension.

## 13. Recommendation Systems

**What it is**: Systems that predict what a user might like based on past behavior or similarity to other users/items.

**How statistics is used**: User-based collaborative filtering relies on correlation (typically Pearson) to measure how similarly two users rate items. Item-based approaches use cosine similarity, which is itself a statistical measure of the angle between two vectors of ratings. Matrix factorization methods decompose the user-item ratings matrix using techniques rooted in linear algebra and statistics, like singular value decomposition.

**Business Example**: A streaming service wants to recommend movies to a user based on how similarly they've rated films compared to other users in the system.

In [15]:
import pandas as pd

ratings = pd.DataFrame({
    'movie_a': [5, 4, 1, 2],
    'movie_b': [4, 5, 2, 1],
    'movie_c': [1, 2, 5, 4],
    'movie_d': [2, 1, 4, 5]
}, index=['user_1', 'user_2', 'user_3', 'user_4'])

user_similarity = ratings.T.corr()
print(user_similarity)

        user_1  user_2  user_3  user_4
user_1     1.0     0.8    -1.0    -0.8
user_2     0.8     1.0    -0.8    -1.0
user_3    -1.0    -0.8     1.0     0.8
user_4    -0.8    -1.0     0.8     1.0


**Interpretation**: user_1 and user_2 correlate strongly (they rate movies similarly) while user_1 and user_3 correlate negatively — that correlation matrix is the statistical backbone that lets the system recommend movie_b to user_1 based on what user_2 liked, without ever needing to know why they're similar.